# 01. Core: Deep Agent + LangGraph State

## Что агент пока не умеет

Мы начинаем не с filesystem и не с Jenkins. Первый слой — это сам agent harness:

```text
user message
→ LangGraph state
→ model
→ agent loop
→ final response
```

На этом этапе агент ещё не знает реальный репозиторий и не должен делать вид, что знает.


## Примитивы Core

```text
Model
решает следующий шаг

State
хранит messages и промежуточные результаты

Agent loop
повторяет model/tool шаги до завершения
```

`create_deep_agent()` здесь не создаёт "магический интеллект". Он собирает готовый Deep Agent harness поверх LangGraph state.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd() / 'workshop_notebooks' / 'openclaw_path'):
    if (candidate / 'workshop_utils.py').exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from workshop_utils import REPO_ROOT, model_name, print_stage_context, register_graphs, workspace_root, write_text

print_stage_context()
print("pyproject:", REPO_ROOT / "pyproject.toml")

from deepagents import create_deep_agent

CORE_PROMPT = """\
You are OpenClaw at stage 01 Core. Respond in Russian by default.
You have no real workspace backend and no external tools yet.
If asked about repository files, say that you cannot inspect the real repository in this graph.
"""

core_agent = create_deep_agent(model=model_name(), tools=[], system_prompt=CORE_PROMPT)
print(type(core_agent).__name__)
print("User tools:", [])
print("Backend:", None)


In [ ]:
ENTRYPOINT = '''\
from __future__ import annotations

import os
from pathlib import Path

from deepagents import create_deep_agent
from dotenv import load_dotenv


DEFAULT_MODEL = "openrouter:tencent/hy3:free"
REPO_ROOT = Path(__file__).resolve().parents[1]

load_dotenv(REPO_ROOT / ".env")


CORE_PROMPT = """\
You are OpenClaw at stage 01 Core.
Respond in the user's language; default to Russian.

This graph demonstrates only the core Deep Agent harness:
- model;
- LangGraph state;
- message history;
- agent loop;
- no real workspace backend;
- no external tools.

If the user asks about repository files or external systems, explain that this
Core graph cannot inspect the real repository or call external APIs yet. Do not guess.
"""


agent = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL),
    tools=[],
    system_prompt=CORE_PROMPT,
)
'''

entrypoint = write_text('agents/openclaw_01_core.py', ENTRYPOINT)
config_path = register_graphs({
    'openclaw_01_core': './agents/openclaw_01_core.py:agent'
})

print("Entrypoint:", entrypoint.relative_to(REPO_ROOT))
print("LangGraph config:", config_path.relative_to(REPO_ROOT))


## Проверка в LangGraph Studio

### Запрос

```text
Найди файл pyproject.toml и назови имя проекта и три зависимости. Если у тебя нет доступа к файлам, прямо скажи об этом.
```

### Ожидаемое поведение

1. `openclaw_01_core` не читает файловую систему.
2. Агент честно говорит, что real workspace backend ещё не подключён.
3. В trace виден core agent loop без полезных external capabilities.

### Что изменилось относительно `00`

История стала исполняемым LangGraph graph.

### Текущее ограничение

Агент имеет state и loop, но пока не может сам добывать контекст из workspace.
